# NTL: OLMo-2-13B two-stage LoRA member, trained on free Colab (T4)

Runs `name-that-llm/lab/09_train_olmo13b.py` (embedded below): nishantdahal's 14B two-stage LoRA recipe on
allenai/OLMo-2-1124-13B (Apache-2.0), first 27 of 40 layers, base in 4-bit NF4 (QLoRA). Units, in this order:

1. `stage1`: 1 epoch of nishantdahal's public pool (pool_v4, CC BY-SA 4.0), about 17 h on one T4, so several sessions.
   It normally runs on Kaggle (2x T4, about 8 h); when it has, it is picked up from the dataset below and skipped.
2. `fold0` .. `fold4`: stage 2 on 4/5 of the prompts, scoring the held-out fifth (the OOF), about 1 h each.
3. `full`: stage 2 on all rows, writes `lora_olmo2_13b/` (the member the 0.636 notebook variant loads), about 1.3 h.
4. `merge`: the OOF summary (`oof_olmo13b.npz`, `summary_olmo13b.json`), seconds.

**What to do:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Approve the Google Drive
mount, and allow notebook access to the `KAGGLE_API_TOKEN` secret (key icon in the left bar) if Colab asks.
Keep the tab open. The first run downloads about 37 GB of base weights (the shards of layers 0-26) to the runtime.

**Resuming.** Training saves a resume point to Google Drive (`MyDrive/ntl_olmo13b/`, about 0.5 GB) every 100 steps,
and every finished unit is kept there. If Colab disconnects, reconnect and **Run all** again: finished units are skipped
and the running one continues from its last save.

**Results** go to the private Kaggle dataset `aryanawasthiteykik/ntl-olmo13b-member` after every unit, together with
what is already there (the Kaggle units' outputs).


In [ ]:
OUTPUTS = "aryanawasthiteykik/ntl-olmo13b-member"   # private; also where Kaggle-finished units come from
UNITS = ["stage1", "fold0", "fold1", "fold2", "fold3", "fold4", "full", "merge"]
WORK = "/content/drive/MyDrive/ntl_olmo13b"   # outputs and resume points, survive disconnects
CODE = "/content/ntl_olmo13b_code"            # the embedded scripts (re-written on every Run all)
DATA = "/content/data"                        # train.csv (competition) and pool_v4.parquet (nishantdahal)
BASE = "/content/olmo2_13b"                   # the base's config, tokenizer and the shards of layers < KEEP
HF_ID, KEEP = "allenai/OLMo-2-1124-13B", 27   # same weights as Kaggle's allenai/olmo-2/Transformers/1124-13b/1
EST_H = {"stage1": 17.0, "fold0": 1.0, "fold1": 1.0, "fold2": 1.0, "fold3": 1.0, "fold4": 1.0, "full": 1.3,
         "merge": 0.0}   # T4 estimates


In [ ]:
# A GPU first, so a CPU runtime fails at once with the fix.
import shutil, subprocess

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True) if shutil.which("nvidia-smi") else None
if gpu is None or gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError("No GPU in this runtime. Runtime → Change runtime type → T4 GPU, then Runtime → Run all again.")
GPU = gpu.stdout.strip().splitlines()[0]
print("GPU:", GPU)


In [ ]:
import json, os, sys, time
from importlib.metadata import PackageNotFoundError, version

from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
except Exception as e:
    raise RuntimeError("Add your Kaggle API token as the Colab secret KAGGLE_API_TOKEN (key icon in the left bar), "
                       "turn on its notebook access, then Run all again.") from e


def ver(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None


need = [] if ver("transformers") == "5.18.0" else ["transformers==5.18.0"]   # the version 09 was checked with
need += [] if ver("bitsandbytes") == "0.50.2" else ["bitsandbytes==0.50.2"]   # the 0.636 notebook's wheel
need += [] if int((ver("kaggle") or "0").split(".")[0]) >= 2 else ["kaggle==2.2.4"]
need += [] if ver("accelerate") else ["accelerate"]
if need:
    print("pip install", *need)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
sh = lambda cmd: subprocess.run(cmd, shell=True, check=True)
os.makedirs(WORK, exist_ok=True)
os.makedirs(DATA, exist_ok=True)
if not os.path.exists(f"{DATA}/train.csv"):
    sh(f"kaggle competitions download name-that-llm -f train.csv -p {DATA}")
    sh(f"cd {DATA} && (unzip -q -o train.csv.zip && rm -f train.csv.zip || true)")
if not os.path.exists(f"{DATA}/pool_v4.parquet"):
    sh(f"kaggle datasets download nishantdahal/ntl-handwriting-blend -f pool_v4.parquet -p {DATA}")
    sh(f"cd {DATA} && (unzip -q -o pool_v4.parquet.zip && rm -f pool_v4.parquet.zip || true)")
# Units already finished (on Kaggle or in an earlier session) come from the dataset; Drive copies win.
if subprocess.run(["kaggle", "datasets", "status", OUTPUTS], capture_output=True).returncode == 0:
    sh(f"kaggle datasets download {OUTPUTS} -p /content/member_ds --unzip -o")
    import zipfile
    for z in [f for f in os.listdir("/content/member_ds") if f.endswith(".zip")]:   # folders uploaded as zips
        zipfile.ZipFile(f"/content/member_ds/{z}").extractall(f"/content/member_ds/{z[:-4]}")
        os.remove(f"/content/member_ds/{z}")
    for f in os.listdir("/content/member_ds"):
        if not os.path.exists(f"{WORK}/{f}"):
            (shutil.copytree if os.path.isdir(f"/content/member_ds/{f}") else shutil.copy)(f"/content/member_ds/{f}",
                                                                                          f"{WORK}/{f}")
# The base: only the shards that hold the embeddings and layers 0..KEEP-1.
from huggingface_hub import hf_hub_download, snapshot_download
idx = json.load(open(hf_hub_download(HF_ID, "model.safetensors.index.json", local_dir=BASE)))["weight_map"]
shards = sorted({v for k, v in idx.items() if k.startswith("model.embed_tokens") or
                 (k.startswith("model.layers.") and int(k.split(".")[2]) < KEEP)})
snapshot_download(HF_ID, local_dir=BASE, allow_patterns=["*.json", "tokenizer*", "vocab*", "merges*", *shards])
print("train.csv, pool_v4.parquet ready | base", BASE, len(shards), "shards | outputs on Drive in", WORK,
      "|", sorted(os.listdir(WORK)))


In [ ]:
# The scripts, embedded verbatim from the repo (name-that-llm/lab); CODE_SHA identifies this version.
import hashlib, os
FILES = {}
FILES['09_train_olmo13b.py'] = '"""Step 9: a two-stage LoRA member on OLMo-2-13B, with prompt-grouped OOF (GPU; QLoRA on T4s).\n\nWhat: nishantdahal\'s 14B two-stage LoRA recipe ("Two-stage LoRA, part two") on allenai/OLMo-2-1124-13B (Apache-2.0):\n- trunk: first 27 of 40 layers (the depth nishant keeps for the same-shape Qwen3-14B, 40 layers x 5120), pooled at\n  layers 2, 4, 6, 9, 13, 17, 22, 27 (his Qwen3-14B layers), mean without the first token, LayerNorm per layer, head\n  dropout 0.1, a plain head and a triplet-centred head; every passage ends with <|endoftext|> (100257), max 447 tokens;\n- LoRA rank 16 / alpha 32 / dropout 0.05 on every linear layer of the trunk; the base is NF4 4-bit (fp16 compute), with\n  embeddings, norms and the residual stream in fp32 (13B in fp16 does not fit a T4 and OLMo overflows fp16);\n- batches of 6 triplets, 3 passages of one prompt each; loss = CE(plain) + CE(centred on triplet-centred features);\n  AdamW wd 0.01, 10% warm-up then cosine, clip 1.0;\n- stage 1: 1 epoch of nishant\'s pool_v4 minus passages with content cosine >= 0.3 to a train passage (07\'s filter),\n  class-weighted, lr 1e-4 (LoRA) / 1e-3 (heads); stage 2: 2 epochs of train, 5e-5 / 5e-4.\nUnits: stage1 (the pool; it never sees train); fold<k> (stage 2 from stage 1 on 4/5 of the prompts, GroupKFold(5) on\nprompt_id = 07\'s folds, scores the held-out fifth); full (stage 2 on all of train: the shipped member); merge (OOF).\nSeveral GPUs: run under torchrun; each step\'s triplets are split over the ranks and the LoRA/head gradients summed.\n\nWhy: a new ensemble member from another model family. The #2 team (0.673) uses OLMo-2-13B; our 0.636 base has only\nQwen decoders. The OOF measures it alone and its value next to the base\'s n-gram SVC, before any submission.\n\nOutput (in --out); a unit whose output exists is skipped (the cache):\n- stage1/lora_heads.safetensors, stage1_train.npz (train scored by the stage-1 model), summary_stage1.json;\n- oof_fold<k>.npz: rows, ids, y, plain and centred logits (centred within each held-out prompt);\n- lora_olmo2_13b/: meta.json, adapter/ (peft layout), heads.safetensors, the layout of nishant\'s 14B members plus\n  cfg["quant"] = "nf4" (score it with olmo13b_member.olmo_features); summary_full.json;\n- oof_olmo13b.npz + summary_olmo13b.json after merge (with --svc-oof: the blend with the SVC, paired bootstrap);\n- state/<unit>.pt: the resume point (every --save-every steps), deleted when the unit is done.\n\nUsage:\n  torchrun --nproc_per_node 2 09_train_olmo13b.py --unit stage1 --train train.csv --pool pool_v4.parquet \\\n      --base OLMO_DIR --out OUT --stage1-hours 8.5\n  python 09_train_olmo13b.py --unit fold0,fold1,fold2 ...   (one GPU each; also fold3,fold4,full; then --unit merge)\n  CPU check: python lab/tests/check_olmo13b.py (a tiny random OLMo-2, end to end, NF4 and fp32, 1 and 2 ranks)\n"""\nimport argparse\nimport importlib.util\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport olmo13b_member as om  # noqa: E402\n\nLABELS = [\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\', \'gemini-3.1-pro-preview\',\n          \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\', \'claude-opus-4-6\', \'claude-opus-4-8\',\n          \'claude-opus-5\', \'claude-sonnet-5\', \'claude-fable-5\', \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\',\n          \'kimi-k3\']   # the part-two notebook\'s head order (meta.json is checked against it there)\nK = len(LABELS)\nMEMBER = \'lora_olmo2_13b\'\n\n\ndef load_step(n):\n    spec = importlib.util.spec_from_file_location(f\'s{n}\', os.path.join(HERE, n))\n    m = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(m)\n    return m\n\n\n# ---------------------------------------------------------------- distributed helpers (one process = one GPU)\nRANK, WORLD = int(os.environ.get(\'RANK\', 0)), int(os.environ.get(\'WORLD_SIZE\', 1))\n\n\ndef log(*a):\n    if RANK == 0:\n        print(time.strftime(\'%H:%M:%S\'), *a, flush=True)\n\n\ndef dist():\n    import torch.distributed as d\n    return d\n\n\ndef triplets(prompts, models, rng):\n    """One epoch: every passage once (up to 2 per prompt left over), in triplets of one prompt, models interleaved\n    so a triplet tends to hold 3 different models (as in colab/ntl_twostage_lora_train.py)."""\n    out = []\n    df = pd.DataFrame(dict(p=prompts, m=models))\n    for _, g in df.groupby(\'p\', sort=True):\n        by_model = {}\n        for i in rng.permutation(g.index.values):\n            by_model.setdefault(models[i], []).append(int(i))\n        order = [v[k] for k in range(max(map(len, by_model.values()))) for v in by_model.values() if k < len(v)]\n        out += [order[s:s + 3] for s in range(0, len(order) - 2, 3)]\n    return [out[i] for i in rng.permutation(len(out))]\n\n\nclass Trainer:\n    def __init__(self, a, dev):\n        self.a, self.dev = a, dev\n        self.tok = om.tokenizer(a.base)\n        self.eos = self.tok.convert_tokens_to_ids(\'<|endoftext|>\')\n        t0 = time.time()\n        self.model = om.load_trunk(a.base, a.keep, a.quant, str(dev), a.compute_dtype)\n        torch.manual_seed(a.seed)   # the same LoRA init on every rank\n        self.lora = om.attach_lora(self.model, a.r, a.alpha, a.lora_dropout)\n        H = self.model.config.hidden_size\n        self.heads = nn.ModuleDict(dict(norms=om.make_norms(H, len(a.layers), dev),\n                                        plain=nn.Linear(H * len(a.layers), K), centered=nn.Linear(H * len(a.layers), K)))\n        self.heads.to(dev)\n        self.hdrop = nn.Dropout(a.head_drop)\n        self.params = [p for m in self.lora.values() for p in (m.A, m.B)]\n        if WORLD > 1:\n            for p in self.params + list(self.heads.parameters()):\n                dist().broadcast(p.data, 0)\n        if dev.type == \'cuda\':\n            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\': False})\n        log(f\'trunk {a.keep} layers ({a.quant or "fp32"}) loaded in {time.time() - t0:.0f}s; LoRA on {len(self.lora)} \'\n            f\'linear layers, {sum(p.numel() for p in self.params) / 1e6:.1f}M params\')\n        if dev.type == \'cuda\':\n            log(f\'memory after load: {torch.cuda.memory_allocated(dev) / 2**30:.1f} GiB\')\n\n    def encode(self, texts):\n        return om.encode(self.tok, texts, self.a.max_len, self.eos)\n\n    # --- weights in and out\n    def state(self):\n        sd = {f\'lora.{k}\': v for k, v in om.adapter_state(self.lora).items()}\n        sd.update({f\'heads.{k}\': v.detach().float().cpu().contiguous() for k, v in self.heads.state_dict().items()})\n        return sd\n\n    @torch.no_grad()\n    def load_state(self, sd):\n        om.load_adapter(self.lora, {k[5:]: v for k, v in sd.items() if k.startswith(\'lora.\')})\n        self.heads.load_state_dict({k[6:]: v for k, v in sd.items() if k.startswith(\'heads.\')})\n\n    # --- training\n    def loss(self, ids, rows, y, w):\n        Z = om.pool(self.model, self.tok, [ids[i] for i in rows], self.a.layers, self.heads[\'norms\'], self.dev)\n        Z = self.hdrop(Z)\n        n = len(rows) // 3\n        Zc = (Z.view(n, 3, -1) - Z.view(n, 3, -1).mean(1, keepdim=True)).view(len(rows), -1)\n        yb = torch.as_tensor(y[rows], device=self.dev)\n        ce = nn.functional.cross_entropy\n        return ce(self.heads[\'plain\'](Z), yb, weight=w) + ce(self.heads[\'centered\'](Zc), yb, weight=w)\n\n    def probe(self, ids, y):\n        """Forward + backward on the longest triplet of micro-batch size, so out-of-memory shows in minute one."""\n        n = 3 * self.a.micro_triplets\n        rows = list(np.argsort([len(x) for x in ids])[-n:])\n        self.model.train()\n        self.loss(ids, rows, y, None).backward()\n        for p in self.params + list(self.heads.parameters()):\n            p.grad = None\n        if self.dev.type == \'cuda\':\n            log(f\'memory probe ok ({n} passages of {len(ids[rows[-1]])} tokens): peak \'\n                f\'{torch.cuda.max_memory_allocated(self.dev) / 2**30:.1f} GiB\')\n\n    def fit(self, ids, y, prompts, models, epochs, lr_lora, lr_head, class_weighted, unit, max_secs=None):\n        a = self.a\n        rng = np.random.default_rng(a.seed + (0 if unit == \'stage1\' else 1))\n        trips = [t for _ in range(epochs) for t in triplets(prompts, models, rng)]\n        S = a.triplets_per_step\n        plan = [trips[s:s + S] for s in range(0, len(trips) - S + 1, S)]\n        w = None\n        if class_weighted:\n            cnt = np.bincount(y, minlength=K).astype(float)\n            w = torch.tensor(np.where(cnt > 0, cnt.sum() / (K * np.maximum(cnt, 1)), 0.0), dtype=torch.float32,\n                             device=self.dev)\n        opt = torch.optim.AdamW([{\'params\': self.params, \'lr\': lr_lora},\n                                 {\'params\': list(self.heads.parameters()), \'lr\': lr_head}], weight_decay=0.01)\n        sch = dict(total=len(plan))\n        warm = max(1, int(0.1 * len(plan)))\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (\n            1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, sch[\'total\'] - warm)))))\n        fp16 = self.dev.type == \'cuda\' and a.quant == \'nf4\' and a.compute_dtype == \'float16\'\n        scaler = torch.amp.GradScaler(\'cuda\', init_scale=2.0 ** 12, enabled=fp16)\n        state_path = os.path.join(a.out, \'state\', f\'{unit}.pt\')\n        start = 0\n        if os.path.exists(state_path):\n            st = torch.load(state_path, map_location=\'cpu\', weights_only=False)\n            self.load_state(st[\'weights\'])\n            opt.load_state_dict(st[\'opt\']), sched.load_state_dict(st[\'sched\']), scaler.load_state_dict(st[\'scaler\'])\n            start, sch[\'total\'] = st[\'step\'], st[\'total\']\n            log(f\'{unit}: resumed at step {start}/{sch["total"]}\')\n        trainable = self.params + list(self.heads.parameters())\n        self.model.train(), self.heads.train()\n        t0, run, skipped, info = time.time(), [], 0, dict(planned=len(plan), passages=3 * S * len(plan))\n        log(f\'{unit}: {len(plan)} steps of {S} triplets, {WORLD} rank(s)\')\n        step = start\n        while step < sch[\'total\']:\n            mine = plan[step][RANK::WORLD]\n            for s in range(0, len(mine), a.micro_triplets):\n                trip = mine[s:s + a.micro_triplets]\n                lo = self.loss(ids, [i for t in trip for i in t], y, w)\n                if not torch.isfinite(lo):   # fp16 overflow on a rare passage: skip that micro-batch\n                    skipped += 1\n                    continue\n                scaler.scale(lo * len(trip) / S).backward()\n                run.append(lo.item())\n            for p in trainable:\n                if p.grad is None:\n                    p.grad = torch.zeros_like(p)\n            if WORLD > 1:   # sum the ranks\' gradients (each already holds its share of the step\'s mean)\n                flat = torch.cat([p.grad.reshape(-1) for p in trainable])\n                dist().all_reduce(flat)\n                o = 0\n                for p in trainable:\n                    p.grad.copy_(flat[o:o + p.numel()].view_as(p))\n                    o += p.numel()\n                del flat\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n            scaler.step(opt), scaler.update(), sched.step()\n            opt.zero_grad(set_to_none=True)\n            step += 1\n            if step == start + 20 and max_secs:   # shorten the schedule to the time budget (cosine still reaches 0)\n                per = (time.time() - t0) / 20\n                new = torch.tensor([min(sch[\'total\'], max(warm + 1, int(max_secs / per)))])\n                if WORLD > 1:\n                    dist().broadcast(new, 0)\n                if int(new) < sch[\'total\']:\n                    log(f\'{unit}: time budget {max_secs / 3600:.1f} h: schedule cut to {int(new)} of {sch["total"]} \'\n                        f\'steps ({per:.1f} s/step)\')\n                sch[\'total\'] = int(new)\n            if step % 20 == 0 or step == sch[\'total\']:\n                el = time.time() - t0\n                log(f\'{unit} step {step}/{sch["total"]} loss {np.mean(run[-60:]) if run else float("nan"):.3f} \'\n                    f\'skipped {skipped} scale {scaler.get_scale():.0f} | {el / 60:.1f} min, \'\n                    f\'ETA {el / (step - start) * (sch["total"] - step) / 60:.0f} min\')\n            if step % a.save_every == 0 and step < sch[\'total\'] and RANK == 0:\n                os.makedirs(os.path.dirname(state_path), exist_ok=True)\n                torch.save(dict(weights=self.state(), opt=opt.state_dict(), sched=sched.state_dict(),\n                                scaler=scaler.state_dict(), step=step, total=sch[\'total\']), state_path + \'.tmp\')\n                os.replace(state_path + \'.tmp\', state_path)   # never a half-written resume point\n        self.model.eval(), self.heads.eval()\n        info.update(steps=sch[\'total\'], secs=time.time() - t0, skipped=skipped,\n                    final_loss=float(np.mean(run[-60:])) if run else None)\n        return info\n\n    # --- scoring\n    def logits(self, ids, rows, groups):\n        """Plain logits and logits of the centred head (features centred within each group) for `rows`.\n        Several ranks: whole groups per rank, gathered on every rank."""\n        my_groups = set(np.unique(groups[rows])[RANK::WORLD])\n        mine = np.array([r for r in rows if groups[r] in my_groups], dtype=int)\n        Z = om.features(self.model, self.tok, [ids[i] for i in mine], self.a.layers, self.heads[\'norms\'],\n                        self.a.eval_bs, self.dev)\n        Zc = Z.copy()\n        for g in np.unique(groups[mine]):\n            sel = groups[mine] == g\n            Zc[sel] -= Zc[sel].mean(0)\n        hw = {k: v.detach().float().cpu().numpy() for k, v in self.heads.state_dict().items()}\n        part = (mine, Z @ hw[\'plain.weight\'].T + hw[\'plain.bias\'], Zc @ hw[\'centered.weight\'].T + hw[\'centered.bias\'])\n        parts = [part]\n        if WORLD > 1:\n            parts = [None] * WORLD\n            dist().all_gather_object(parts, part)\n        pos = {r: i for i, r in enumerate(rows)}\n        P, C = np.zeros((len(rows), K), np.float32), np.zeros((len(rows), K), np.float32)\n        for m, p, c in parts:\n            idx = [pos[r] for r in m]\n            P[idx], C[idx] = p, c\n        return P, C\n\n\ndef bacc(y, pred):\n    from sklearn.metrics import balanced_accuracy_score\n    return float(balanced_accuracy_score(y, pred))\n\n\ndef export(tr, a, out_dir, info):\n    """nishant\'s checkpoint layout: meta.json, adapter/ (peft keys), heads.safetensors."""\n    from safetensors.torch import save_file\n    os.makedirs(os.path.join(out_dir, \'adapter\'), exist_ok=True)\n    save_file(om.adapter_state(tr.lora), os.path.join(out_dir, \'adapter\', \'adapter_model.safetensors\'))\n    json.dump({\'r\': a.r, \'lora_alpha\': a.alpha, \'lora_dropout\': a.lora_dropout, \'peft_type\': \'LORA\', \'bias\': \'none\',\n               \'base_model_name_or_path\': \'allenai/OLMo-2-1124-13B\', \'auto_mapping\': {\'base_model_class\': \'Olmo2Model\'},\n               \'target_modules\': sorted({k.split(\'.\')[-1] for k in tr.lora})},\n              open(os.path.join(out_dir, \'adapter\', \'adapter_config.json\'), \'w\'), indent=1)\n    save_file({k: v.detach().float().cpu().contiguous() for k, v in tr.heads.state_dict().items()},\n              os.path.join(out_dir, \'heads.safetensors\'))\n    cfg = dict(model_dir=\'allenai/OLMo-2-1124-13B\', keep=a.keep, layers=a.layers, max_len=a.max_len, r=a.r,\n               alpha=a.alpha, ldrop=a.lora_dropout, head_drop=a.head_drop, lr_adapter=a.lr2[0], lr_head=a.lr2[1],\n               wd=0.01, warmup=0.1, epochs=float(a.stage2_epochs), triplets_per_step=a.triplets_per_step, group=3,\n               eval_bs=a.eval_bs, seed=a.seed, grad_ckpt=True, eos_id=int(tr.eos),\n               eos_note=\'eos_id is the end token appended in training (<|endoftext|>)\',\n               quant=a.quant, compute_dtype=a.compute_dtype, nonquant_dtype=\'float32\', final_norm=False,\n               stage1=dict(epochs=a.stage1_epochs, lr_adapter=a.lr1[0], lr_head=a.lr1[1], class_weighted=True,\n                           pool=\'nishantdahal/ntl-handwriting-blend pool_v4 minus content cosine >= 0.3 to train\'))\n    json.dump(dict(cfg=cfg, labels=LABELS, train_info=info), open(os.path.join(out_dir, \'meta.json\'), \'w\'), indent=1)\n\n\ndef merge(a, train, y, folds):\n    rows, P, C = np.arange(len(train)), np.zeros((len(train), K)), np.zeros((len(train), K))\n    for f in range(a.folds):\n        z = np.load(os.path.join(a.out, f\'oof_fold{f}.npz\'))\n        P[z[\'rows\']], C[z[\'rows\']] = z[\'plain\'], z[\'centred\']\n    s08 = load_step(\'08_blend_oof.py\')\n    Pp, Pc = s08.softmax(P), s08.softmax(C)\n    st1 = np.load(os.path.join(a.out, \'stage1_train.npz\'))\n    res = dict(oof_plain=bacc(y, P.argmax(1)), oof_centred_by_prompt=bacc(y, C.argmax(1)),\n               oof_plain_sinkhorn=bacc(y, s08.blend([(1.0, Pp)])),\n               oof_plain_plus_twin=bacc(y, s08.blend([(1.0, Pp), (0.25, Pc)])),\n               stage1_only=bacc(y, st1[\'plain\'].argmax(1)),\n               fold_plain=[bacc(y[folds == f], P[folds == f].argmax(1)) for f in range(a.folds)])\n    np.savez(os.path.join(a.out, \'oof_olmo13b.npz\'), ids=train.id.astype(str).to_numpy(dtype=str), fold=folds, y=y,\n             labels=np.array(LABELS, dtype=str), plain=P.astype(np.float32), centred=C.astype(np.float32))\n    if a.svc_oof:   # value next to the base\'s SVC (weight 1.0 there); the LoRAs get 0.2-0.4 in the base\n        z = np.load(a.svc_oof)\n        assert list(z[\'ids\']) == train.id.astype(str).tolist() and (z[\'fold\'] == folds).all()\n        Ps = s08.softmax(z[\'scores\'])   # 08\'s oof_svc.npz, alphabetical label order -> ours\n        Ps = Ps[:, [s08.LABELS.index(l) for l in LABELS]]\n        base = s08.blend([(1.0, Ps)])\n        res[\'svc_alone_sinkhorn\'] = bacc(y, base)\n        groups, ug = train.prompt_id.values, np.unique(train.prompt_id.values)\n        gidx = {g: np.where(groups == g)[0] for g in ug}\n        rng = np.random.default_rng(0)\n        boots = [np.concatenate([gidx[g] for g in rng.choice(ug, len(ug))]) for _ in range(a.boot)]\n        for wgt in (0.1, 0.2, 0.4):\n            pr = s08.blend([(1.0, Ps), (wgt, Pp)])\n            d = np.array([bacc(y[i], pr[i]) - bacc(y[i], base[i]) for i in boots])\n            res[f\'svc_plus_olmo_{wgt}_sinkhorn\'] = bacc(y, pr)\n            res[f\'svc_plus_olmo_{wgt}_p_better\'] = float((d > 0).mean())\n    side = {f[8:-5]: json.load(open(os.path.join(a.out, f))) for f in (\'summary_stage1.json\', \'summary_full.json\')\n            if os.path.exists(os.path.join(a.out, f))}\n    json.dump(dict(scores=res, **side), open(os.path.join(a.out, \'summary_olmo13b.json\'), \'w\'), indent=1)\n    for k, v in res.items():\n        log(f\'{k:36s} {v if isinstance(v, list) else round(v, 4)}\')\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--unit\', required=True, help=\'stage1, fold<k>, full, merge, or a comma list\')\n    ap.add_argument(\'--train\', required=True)\n    ap.add_argument(\'--pool\')\n    ap.add_argument(\'--base\', help=\'OLMo-2-1124-13B dir (config, tokenizer, safetensors)\')\n    ap.add_argument(\'--out\', required=True)\n    ap.add_argument(\'--quant\', default=\'nf4\', type=lambda s: None if s == \'none\' else s, help=\'nf4 or none\')\n    ap.add_argument(\'--compute-dtype\', default=\'float16\')\n    ap.add_argument(\'--keep\', type=int, default=27)\n    ap.add_argument(\'--layers\', default=\'2,4,6,9,13,17,22,27\', type=lambda s: [int(x) for x in s.split(\',\')])\n    ap.add_argument(\'--max-len\', type=int, default=447)\n    ap.add_argument(\'--r\', type=int, default=16)\n    ap.add_argument(\'--alpha\', type=int, default=32)\n    ap.add_argument(\'--lora-dropout\', type=float, default=0.05)\n    ap.add_argument(\'--head-drop\', type=float, default=0.1)\n    ap.add_argument(\'--triplets-per-step\', type=int, default=6)\n    ap.add_argument(\'--micro-triplets\', type=int, default=1, help=\'triplets per forward pass (memory only)\')\n    ap.add_argument(\'--stage1-epochs\', type=int, default=1)\n    ap.add_argument(\'--stage2-epochs\', type=int, default=2)\n    ap.add_argument(\'--lr1\', default=\'1e-4,1e-3\', type=lambda s: [float(x) for x in s.split(\',\')])\n    ap.add_argument(\'--lr2\', default=\'5e-5,5e-4\', type=lambda s: [float(x) for x in s.split(\',\')])\n    ap.add_argument(\'--stage1-hours\', type=float, default=None, help=\'cap on stage-1 wall time (shortens the schedule)\')\n    ap.add_argument(\'--eval-bs\', type=int, default=8)\n    ap.add_argument(\'--save-every\', type=int, default=100)\n    ap.add_argument(\'--folds\', type=int, default=5)\n    ap.add_argument(\'--seed\', type=int, default=42)\n    ap.add_argument(\'--svc-oof\', help="08\'s oof_svc.npz, for the blend check in merge")\n    ap.add_argument(\'--boot\', type=int, default=2000)\n    ap.add_argument(\'--limit-pool\', type=int, default=None, help=\'random whole prompts, up to n passages (smoke tests)\')\n    ap.add_argument(\'--limit-prompts\', type=int, default=None, help=\'first n train prompts only (smoke tests)\')\n    ap.add_argument(\'--allow-cpu\', action=\'store_true\')\n    a = ap.parse_args()\n    os.makedirs(a.out, exist_ok=True)\n    assert a.layers[-1] <= a.keep and a.triplets_per_step % WORLD == 0\n    units = a.unit.split(\',\')\n    if not torch.cuda.is_available() and not a.allow_cpu and units != [\'merge\']:\n        raise SystemExit(\'no CUDA GPU in this session (pass --allow-cpu only for smoke tests)\')\n    dev = torch.device(f\'cuda:{int(os.environ.get("LOCAL_RANK", 0))}\' if torch.cuda.is_available() else \'cpu\')\n    if WORLD > 1:\n        if dev.type == \'cuda\':\n            torch.cuda.set_device(dev)\n        dist().init_process_group(\'nccl\' if dev.type == \'cuda\' else \'gloo\')\n        assert units == [\'stage1\'], \'several ranks only for stage1; run stage-2 units one GPU each\'\n\n    train = pd.read_csv(a.train)\n    train[\'text\'] = train[\'text\'].fillna(\'\').astype(str)\n    if a.limit_prompts:\n        train = train[train.prompt_id.isin(sorted(train.prompt_id.unique())[:a.limit_prompts])].reset_index(drop=True)\n    y = train.model.map({l: i for i, l in enumerate(LABELS)}).values\n    from sklearn.model_selection import GroupKFold\n    folds = np.zeros(len(train), int)\n    for f, (_, va) in enumerate(GroupKFold(n_splits=a.folds).split(train, groups=train.prompt_id)):\n        folds[va] = f\n    groups = train.prompt_id.astype(str).values\n    tr_obj, s1_sd = None, None\n\n    for unit in units:\n        done = dict(stage1=\'summary_stage1.json\', full=\'summary_full.json\', merge=\'summary_olmo13b.json\').get(\n            unit, f\'oof_{unit}.npz\')\n        if os.path.exists(os.path.join(a.out, done)):\n            log(f\'{unit}: cached ({done})\')\n            continue\n        if unit == \'merge\':\n            if RANK == 0:\n                merge(a, train, y, folds)\n            continue\n        T0 = time.time()\n        if tr_obj is None:\n            tr_obj = Trainer(a, dev)\n            ids_tr = tr_obj.encode(train.text)\n        tr = tr_obj\n        if unit == \'stage1\':\n            pool = pd.read_parquet(a.pool)\n            pool = pool[pool.model.isin(LABELS)].reset_index(drop=True)\n            keep = load_step(\'07_train_encoders.py\').leak_filter(pool.text.tolist(), train.text.tolist()) & \\\n                ~pool.text.isin(set(train.text))\n            log(f\'pool {len(pool)}, kept {int(keep.sum())} after the train-similarity filter\')\n            pool = pool[keep].reset_index(drop=True)\n            if a.limit_pool:\n                ps = pool.prompt_id.drop_duplicates().sample(frac=1, random_state=0)   # whole prompts, for triplets\n                n = pool.groupby(\'prompt_id\').size()[ps].cumsum()\n                pool = pool[pool.prompt_id.isin(n.index[n <= a.limit_pool])].reset_index(drop=True)\n            ids_pool = tr.encode(pool.text)\n            yp = pool.model.map({l: i for i, l in enumerate(LABELS)}).values\n            tr.probe(ids_pool, yp)\n            info = tr.fit(ids_pool, yp, pool.prompt_id.astype(str).values, pool.model.values, a.stage1_epochs,\n                          *a.lr1, True, unit, max_secs=a.stage1_hours * 3600 if a.stage1_hours else None)\n            P, C = tr.logits(ids_tr, np.arange(len(train)), groups)\n            if RANK == 0:\n                from safetensors.torch import save_file\n                os.makedirs(os.path.join(a.out, \'stage1\'), exist_ok=True)\n                save_file(tr.state(), os.path.join(a.out, \'stage1\', \'lora_heads.safetensors\'))\n                np.savez(os.path.join(a.out, \'stage1_train.npz\'), ids=train.id.astype(str).to_numpy(dtype=str), y=y,\n                         plain=P, centred=C)\n                summ = dict(unit=unit, pool_used=len(pool), info=info, stage1_only_bacc_plain=bacc(y, P.argmax(1)),\n                            stage1_only_bacc_centred_by_prompt=bacc(y, C.argmax(1)), secs=time.time() - T0,\n                            ranks=WORLD, device=torch.cuda.get_device_name(dev) if dev.type == \'cuda\' else \'cpu\',\n                            torch=torch.__version__, transformers=__import__(\'transformers\').__version__)\n                json.dump(summ, open(os.path.join(a.out, \'summary_stage1.json\'), \'w\'), indent=1, default=float)\n                log(f\'stage 1 only, on train: plain {summ["stage1_only_bacc_plain"]:.4f}, centred by prompt \'\n                    f\'{summ["stage1_only_bacc_centred_by_prompt"]:.4f}\')\n        else:\n            from safetensors.torch import load_file\n            if s1_sd is None:\n                s1_sd = load_file(os.path.join(a.out, \'stage1\', \'lora_heads.safetensors\'))\n            tr.load_state(s1_sd)\n            fit_rows = np.arange(len(train)) if unit == \'full\' else np.where(folds != int(unit[4:]))[0]\n            sub = lambda v: v[fit_rows]\n            info = tr.fit([ids_tr[i] for i in fit_rows], sub(y), sub(groups), sub(train.model.values),\n                          a.stage2_epochs, *a.lr2, False, unit)\n            if unit == \'full\':\n                out_dir = os.path.join(a.out, MEMBER)\n                export(tr, a, out_dir, info)\n                chk = np.arange(min(128, len(train)))\n                P_mem, _ = tr.logits(ids_tr, chk, groups)\n                tr_obj = tr = None   # free the training trunk: the reload check loads a second one\n                import gc\n                gc.collect()\n                if dev.type == \'cuda\':\n                    torch.cuda.empty_cache()\n                Z, heads = om.olmo_features(out_dir, a.base, train.text.values[chk].tolist(), a.eval_bs, LABELS,\n                                            device=str(dev))\n                P_re = Z @ heads[\'plain.weight\'].T + heads[\'plain.bias\']\n                agree, diff = float((P_re.argmax(1) == P_mem.argmax(1)).mean()), float(np.abs(P_re - P_mem).max())\n                log(f\'reloaded {out_dir} with olmo_features: top-1 agreement {agree:.3f}, max |logit diff| {diff:.2e}\')\n                assert agree >= 0.98, agree\n                json.dump(dict(unit=unit, info=info, reload_agreement=agree, reload_max_logit_diff=diff,\n                               in_sample_bacc_128=bacc(y[chk], P_re.argmax(1)), secs=time.time() - T0,\n                               device=torch.cuda.get_device_name(dev) if dev.type == \'cuda\' else \'cpu\',\n                               torch=torch.__version__, transformers=__import__(\'transformers\').__version__),\n                          open(os.path.join(a.out, \'summary_full.json\'), \'w\'), indent=1, default=float)\n            else:\n                va = np.where(folds == int(unit[4:]))[0]\n                P, C = tr.logits(ids_tr, va, groups)\n                np.savez(os.path.join(a.out, f\'oof_{unit}.npz\'), rows=va, ids=train.id.values[va].astype(str), y=y[va],\n                         fold=int(unit[4:]), plain=P, centred=C, labels=np.array(LABELS, dtype=str), info=json.dumps(info))\n                log(f\'{unit}: held-out balanced accuracy plain {bacc(y[va], P.argmax(1)):.4f}, centred by prompt \'\n                    f\'{bacc(y[va], C.argmax(1)):.4f} on {len(va)} rows ({(time.time() - T0) / 60:.0f} min)\')\n        st = os.path.join(a.out, \'state\', f\'{unit}.pt\')\n        if RANK == 0 and os.path.exists(st):\n            os.remove(st)\n    if WORLD > 1:\n        dist().barrier()\n        dist().destroy_process_group()\n\n\nif __name__ == \'__main__\':\n    main()\n'
FILES['olmo13b_member.py'] = '"""OLMo-2-13B LoRA member: trunk loading, the LoRA wrapper and pooled features, shared by training and inference.\n\nWhat: the first `keep` decoder layers of allenai/OLMo-2-1124-13B (Apache-2.0), base weights 4-bit NF4 (fp16 compute) with\nembeddings, norms and the residual stream in fp32, final norm dropped; LoRA kept unmerged (y = W x + B A x alpha / r);\nfeatures = mean hidden state (first token left out) at the pooled layers, one LayerNorm each, as in nishantdahal\'s 14B\nmembers. `olmo_features` is the inference entry point; its contract is theirs (meta.json, adapter/ in peft key layout,\nheads.safetensors), plus cfg["quant"].\nWhy: 09_train_olmo13b.py trains with this module and the 0.636 notebook variant embeds it verbatim, so test features\nare computed by the same code and the same NF4 base as in training. Merging into an fp16 base, as nishant\'s code\ndoes, would read a base the heads never saw (the NF4 rounding is not in the adapter).\nOutput: none (a module). Usage: from olmo13b_member import olmo_features; Z, heads = olmo_features(ckpt, base, texts,\n8, LABELS); check: lab/tests/check_olmo13b.py.\n"""\nimport gc\nimport json\nimport math\nimport os\n\nimport numpy as np\nimport torch\nfrom torch import nn\n\nOLMO_SPEC = dict(model_type=\'olmo2\', hidden_size=5120, num_hidden_layers=40)   # find_base() spec in the notebook\n\n\nclass LoRALinear(nn.Module):\n    """y = W x + (B A dropout(x)) * alpha / r, with W frozen (nn.Linear or bnb Linear4bit); A and B in fp32."""\n\n    def __init__(self, base, r, alpha, dropout):\n        super().__init__()\n        self.base = base\n        dev = base.weight.device\n        self.A = nn.Parameter(torch.empty(r, base.in_features, device=dev))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=dev))\n        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))\n        self.drop, self.scale = nn.Dropout(dropout), alpha / r\n\n    def forward(self, x):\n        return self.base(x) + (self.drop(x).to(self.A.dtype) @ self.A.T @ self.B.T).to(x.dtype) * self.scale\n\n\ndef load_trunk(base_dir, keep, quant, device, compute_dtype=\'float16\'):\n    """The first `keep` layers; quant=\'nf4\' quantizes every linear layer, everything else stays fp32."""\n    from transformers import AutoConfig, AutoModel\n    mc = AutoConfig.from_pretrained(base_dir)\n    mc.num_hidden_layers, mc.use_cache = keep, False\n    if getattr(mc, \'layer_types\', None) is not None:\n        mc.layer_types = mc.layer_types[:keep]\n    kw = dict(config=mc, dtype=torch.float32)\n    if quant == \'nf4\':\n        from transformers import BitsAndBytesConfig\n        kw[\'quantization_config\'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type=\'nf4\',\n                                                       bnb_4bit_compute_dtype=getattr(torch, compute_dtype),\n                                                       bnb_4bit_use_double_quant=False)\n        kw[\'device_map\'] = {\'\': device}\n    else:\n        assert quant is None, quant\n    model = AutoModel.from_pretrained(base_dir, **kw)\n    if quant is None:\n        model.to(device)\n    assert len(model.layers) == keep\n    model.norm = nn.Identity()   # features read the raw residual stream, as in the 14B members\n    model.requires_grad_(False)\n    return model\n\n\ndef attach_lora(model, r, alpha, dropout):\n    """Wrap every linear layer of the decoder layers; returns {module name: LoRALinear} (names as peft keys use)."""\n    lora = {}\n    for name, m in list(model.named_modules()):\n        for child, sub in list(m.named_children()):\n            if isinstance(sub, nn.Linear) and name.startswith(\'layers.\'):\n                setattr(m, child, LoRALinear(sub, r, alpha, dropout))\n                lora[f\'{name}.{child}\'] = getattr(m, child)\n    return lora\n\n\ndef adapter_state(lora):\n    """peft key layout, as nishant\'s merge_lora reads it."""\n    sd = {}\n    for k, m in lora.items():\n        sd[f\'base_model.model.{k}.lora_A.weight\'] = m.A.detach().float().cpu().contiguous()\n        sd[f\'base_model.model.{k}.lora_B.weight\'] = m.B.detach().float().cpu().contiguous()\n    return sd\n\n\n@torch.no_grad()\ndef load_adapter(lora, sd):\n    used = 0\n    for k, m in lora.items():\n        m.A.copy_(sd[f\'base_model.model.{k}.lora_A.weight\'])\n        m.B.copy_(sd[f\'base_model.model.{k}.lora_B.weight\'])\n        used += 2\n    assert used == len(sd), f\'adapter has {len(sd)} tensors, the trunk takes {used}\'\n\n\ndef make_norms(H, n, device):\n    return nn.ModuleList([nn.LayerNorm(H) for _ in range(n)]).to(device).float()\n\n\ndef encode(tok, texts, max_len, eos_id):\n    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=max_len)[\'input_ids\']\n    return [x + [eos_id] for x in ids]\n\n\ndef pool(model, tok, seqs, layers, norms, device):\n    """Pooled features of a batch of token lists: mean over tokens (first one left out), LayerNorm per layer."""\n    enc = tok.pad({\'input_ids\': seqs}, return_tensors=\'pt\').to(device)\n    hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states\n    m = enc[\'attention_mask\'].unsqueeze(-1).float()\n    m[:, 0] = 0   # the first token soaks up attention whatever it is\n    den = m.sum(1).clamp(min=1)\n    return torch.cat([nm((hs[l].float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1)\n\n\ndef tokenizer(base_dir):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(base_dir)\n    tok.padding_side = \'right\'\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\n@torch.no_grad()\ndef features(model, tok, ids, layers, norms, bs, device):\n    """Z for a list of token lists, length-sorted batches; a non-finite batch is redone one passage at a time."""\n    order = np.argsort([len(x) for x in ids], kind=\'stable\')\n    Z = np.zeros((len(ids), norms[0].normalized_shape[0] * len(layers)), np.float32)\n    for s in range(0, len(order), bs):\n        idx = order[s:s + bs]\n        z = pool(model, tok, [ids[i] for i in idx], layers, norms, device)\n        if not torch.isfinite(z).all():\n            z = torch.cat([pool(model, tok, [ids[i]], layers, norms, device) for i in idx])\n            assert torch.isfinite(z).all(), \'non-finite features even one passage at a time\'\n        Z[idx] = z.cpu().numpy()\n    return Z\n\n\n@torch.no_grad()\ndef olmo_features(ckpt_dir, base_dir, texts, bs, labels, device=\'cuda:0\'):\n    """Pooled features Z (one row per passage) and the checkpoint\'s heads, like the notebook\'s lora_features."""\n    from safetensors.torch import load_file\n    meta = json.load(open(os.path.join(ckpt_dir, \'meta.json\')))\n    assert meta[\'labels\'] == list(labels), "label order differs from the checkpoint\'s heads"\n    cfg = meta[\'cfg\']\n    tok = tokenizer(base_dir)\n    model = load_trunk(base_dir, cfg[\'keep\'], cfg[\'quant\'], device, cfg[\'compute_dtype\'])\n    acfg = json.load(open(os.path.join(ckpt_dir, \'adapter\', \'adapter_config.json\')))\n    lora = attach_lora(model, acfg[\'r\'], acfg[\'lora_alpha\'], 0.0)\n    load_adapter(lora, load_file(os.path.join(ckpt_dir, \'adapter\', \'adapter_model.safetensors\')))\n    model.eval()\n    heads = load_file(os.path.join(ckpt_dir, \'heads.safetensors\'))\n    norms = make_norms(model.config.hidden_size, len(cfg[\'layers\']), device)\n    norms.load_state_dict({k[len(\'norms.\'):]: v for k, v in heads.items() if k.startswith(\'norms.\')})\n    eos_id = cfg[\'eos_id\']\n    print(f"OLMo trunk of {cfg[\'keep\']} layers ({cfg[\'quant\']}), end token {eos_id}, tokenizer\'s {tok.eos_token_id}",\n          flush=True)\n    Z = features(model, tok, encode(tok, texts, cfg[\'max_len\'], eos_id), cfg[\'layers\'], norms, bs, device)\n    del model, lora\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith(\'norms.\')}\n'
FILES['07_train_encoders.py'] = '"""Step 7: train the two encoder members ("OURS") of the 0.624-0.627 NTL notebooks, with prompt-grouped OOF (GPU).\n\nWhat: one member per run, either ModernBERT-large or the desklib DeBERTa-v3-large AI-detector, fine-tuned in two stages\nas in nuridudak\'s write-up (see encoder_rebuild_spec.md).\n- Stage 1: the public pool (nishantdahal\'s pool_v4, minus passages close to a train passage), class-weighted CE.\n- Stage 2: the official train.\nCV is GroupKFold(5) on prompt_id. Stage 1 runs once (it never sees train), each fold runs stage 2 from the stage-1 weights\nand predicts its held-out prompts, and the shipped model is stage 2 on the whole train.\n\nWhy: the members\' dataset nuridudak/ntl-ours-v1 went private, so those notebooks drop to the 0.620 base. This rebuilds\nthe members as our own work and measures them out of fold before anything is submitted.\n\nOutput (in --out):\n- <member>/ is an AutoModelForSequenceClassification dir (fp16 safetensors, tokenizer, id2label alphabetical), the layout\n  that enc_infer.py in the notebooks loads.\n- oof_<member>.npz holds ids, prompt_id, fold, labels, y, oof logits and stage1 logits (train predicted by the\n  stage-1-only model).\n- summary_<member>.json holds balanced accuracies and timings.\nAn existing summary_<member>.json means the member is done and the run is skipped (the cache).\n\nUsage:\n  python 07_train_encoders.py --member modernbert --train train.csv --pool pool_v4.parquet --out OUT\n  python 07_train_encoders.py --member desklib    --train train.csv --pool pool_v4.parquet --out OUT\n  CPU smoke test: add --allow-cpu --backbone <tiny HF id or dir> --limit-pool 200 --stage1-epochs 1 --stage2-epochs 1\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics import balanced_accuracy_score\nfrom sklearn.model_selection import GroupKFold\n\nLABELS = sorted([\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\', \'gemini-3.1-pro-preview\',\n                 \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\', \'claude-opus-4-6\', \'claude-opus-4-8\',\n                 \'claude-opus-5\', \'claude-sonnet-5\', \'claude-fable-5\', \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\',\n                 \'kimi-k3\'])   # alphabetical, as in nuridudak\'s heads; the notebooks remap by name anyway\nK = len(LABELS)\nMAX_LEN = 512   # enc_infer.py hard-codes 512\nMEMBERS = {   # published recipe; micro = micro-batch for gradient accumulation (memory only, same maths)\n    \'modernbert\': dict(backbone=\'answerdotai/ModernBERT-large\', lr1=2e-5, bs1=32, micro=8, ckpt=False),   # 16 OOMs a T4\n    \'desklib\': dict(backbone=\'desklib/ai-text-detector-v1.01\', lr1=2.8e-5, bs1=64, micro=16, ckpt=True),\n}\nLEAK_COS = 0.3   # Nishant\'s filter: pool passages with content TF-IDF cosine >= 0.3 to a train passage are out\n\n\ndef log(*a):\n    print(time.strftime(\'%H:%M:%S\'), *a, flush=True)\n\n\ndef leak_filter(pool_texts, train_texts):\n    """Boolean mask of pool passages kept: max content cosine to any train passage below LEAK_COS."""\n    vec = TfidfVectorizer(stop_words=\'english\', sublinear_tf=True).fit(list(train_texts) + list(pool_texts))\n    P, T = vec.transform(pool_texts), vec.transform(train_texts)\n    best = np.zeros(P.shape[0], np.float32)\n    for s in range(0, P.shape[0], 5000):\n        best[s:s + 5000] = (P[s:s + 5000] @ T.T).max(1).toarray().ravel()\n    return best < LEAK_COS\n\n\ndef build_model(backbone, member):\n    """The classifier at its starting point, and the backbone\'s tokenizer."""\n    from safetensors.torch import load_file\n    from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer\n    path = backbone\n    if not os.path.isdir(path):\n        from huggingface_hub import snapshot_download\n        path = snapshot_download(backbone, allow_patterns=[\'*.json\', \'*.safetensors\', \'*.model\', \'*.txt\'],\n                                 ignore_patterns=[\'onnx/*\'])\n    kw = dict(num_labels=K, id2label=dict(enumerate(LABELS)), label2id={l: i for i, l in enumerate(LABELS)})\n    cfg = AutoConfig.from_pretrained(path, **kw)\n    cfg.architectures = None\n    if cfg.model_type == \'modernbert\':\n        cfg.classifier_pooling = \'mean\'   # as in nuridudak\'s config.json\n    st = os.path.join(path, \'model.safetensors\')\n    sd = load_file(st) if os.path.exists(st) else None\n    if cfg.model_type == \'deberta-v2\' and sd is not None and any(k.startswith(\'model.\') for k in sd):\n        # desklib stores its encoder under model.* with a 1-logit classifier: rename to deberta.*, start a fresh head\n        model = AutoModelForSequenceClassification.from_config(cfg)\n        sd = {\'deberta.\' + k[len(\'model.\'):]: v for k, v in sd.items() if k.startswith(\'model.\')}\n        missing, unexpected = model.load_state_dict(sd, strict=False)\n        assert not unexpected and all(k.startswith((\'pooler.\', \'classifier.\')) for k in missing), (missing, unexpected)\n        log(f\'desklib encoder loaded, fresh head: {missing}\')\n    else:\n        model = AutoModelForSequenceClassification.from_pretrained(path, config=cfg, ignore_mismatched_sizes=True)\n    tok = AutoTokenizer.from_pretrained(path)\n    if MEMBERS[member][\'ckpt\']:\n        model.gradient_checkpointing_enable()\n    return model, tok\n\n\ndef encode(tok, texts):\n    return tok(list(texts), truncation=True, max_length=MAX_LEN)[\'input_ids\']   # exactly enc_infer.py\'s call\n\n\ndef amp_dtype(dev):\n    if dev.type != \'cuda\':\n        return None\n    return torch.bfloat16 if torch.cuda.get_device_capability(dev)[0] >= 8 else torch.float16\n\n\ndef batches(ids, bs, rng):\n    """Length-grouped batches: shuffle, sort within chunks of 50 batches, shuffle the batches."""\n    order = rng.permutation(len(ids))\n    out = []\n    for s in range(0, len(order), bs * 50):\n        chunk = sorted(order[s:s + bs * 50], key=lambda i: len(ids[i]))\n        out += [chunk[j:j + bs] for j in range(0, len(chunk), bs)]\n    return [out[i] for i in rng.permutation(len(out))]\n\n\ndef fit(model, tok, ids, y, epochs, lr, bs, micro, dev, class_weight=None, max_secs=None, seed=42, tag=\'\'):\n    """AdamW (wd 0.01), 10% linear warm-up then cosine to 0, grad clip 1.0. If max_secs is set and the run would\n    overrun it, the schedule is shortened after 30 steps (the cosine still reaches 0)."""\n    rng = np.random.default_rng(seed)\n    torch.manual_seed(seed)\n    plan = [b for _ in range(epochs) for b in batches(ids, bs, rng)]\n    state = dict(total=len(plan))\n    warm = max(1, int(0.1 * state[\'total\']))\n\n    def factor(s):\n        if s < warm:\n            return (s + 1) / warm\n        return 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / max(1, state[\'total\'] - warm))))\n\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, factor)\n    dt = amp_dtype(dev)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=dt == torch.float16)\n    w = None if class_weight is None else torch.tensor(class_weight, dtype=torch.float32, device=dev)\n    lossf = torch.nn.CrossEntropyLoss(weight=w, reduction=\'sum\')\n    yt = torch.tensor(y, device=dev)\n    model.train()\n    t0, run, nrun = time.time(), 0.0, 0\n    for step, b in enumerate(plan):\n        if step >= state[\'total\']:\n            break\n        denom = float(w[yt[b]].sum()) if w is not None else float(len(b))\n        for m in range(0, len(b), micro):\n            mb = b[m:m + micro]\n            enc = tok.pad({\'input_ids\': [ids[i] for i in mb]}, return_tensors=\'pt\').to(dev)\n            with torch.autocast(dev.type, dtype=dt, enabled=dt is not None):\n                logits = model(**enc).logits.float()\n            loss = lossf(logits, yt[mb]) / denom\n            scaler.scale(loss).backward()\n            run += loss.item() * denom / len(b)\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n        sched.step()\n        if step == 30 and max_secs:\n            per = (time.time() - t0) / 31\n            if per * state[\'total\'] > max_secs:\n                state[\'total\'] = max(warm + 1, int(max_secs / per))\n                log(f\'{tag} time budget: schedule cut to {state["total"]} of {len(plan)} steps ({per:.2f}s/step)\')\n        nrun += 1\n        if (step + 1) % 200 == 0 or step + 1 == state[\'total\']:\n            log(f\'{tag} step {step + 1}/{state["total"]} loss {run / nrun:.4f} {time.time() - t0:.0f}s\')\n            run, nrun = 0.0, 0\n    model.eval()\n    return dict(steps=min(state[\'total\'], len(plan)), planned=len(plan), secs=time.time() - t0)\n\n\n@torch.inference_mode()\ndef predict(model, tok, ids, dev):\n    """Raw logits, the way enc_infer.py computes them: length-sorted batches of 64, fp16 autocast, fp32 retry."""\n    model.eval()\n    order = np.argsort([len(x) for x in ids])\n    out = np.zeros((len(ids), K), np.float32)\n    dt = amp_dtype(dev)\n    for s in range(0, len(order), 64):\n        idx = order[s:s + 64]\n        b = tok.pad({\'input_ids\': [ids[i] for i in idx]}, return_tensors=\'pt\').to(dev)\n        with torch.autocast(dev.type, dtype=torch.float16 if dt else None, enabled=dt is not None):\n            lg = model(**b).logits.float()\n        if not torch.isfinite(lg).all():\n            lg = model(**b).logits.float()\n        out[idx] = lg.cpu().numpy()\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--member\', choices=list(MEMBERS), required=True)\n    ap.add_argument(\'--train\', required=True)\n    ap.add_argument(\'--pool\', required=True)\n    ap.add_argument(\'--out\', required=True)\n    ap.add_argument(\'--backbone\', help=\'override the HF id / local dir (tiny models for the CPU smoke test)\')\n    ap.add_argument(\'--stage1-epochs\', type=int, default=2)\n    ap.add_argument(\'--stage2-epochs\', type=int, default=2)\n    ap.add_argument(\'--stage2-lr\', type=float, default=1e-5)\n    ap.add_argument(\'--stage2-bs\', type=int, default=32)\n    ap.add_argument(\'--folds\', type=int, default=5)\n    ap.add_argument(\'--stage1-hours\', type=float, default=None, help=\'cap on stage 1 wall time (shortens the schedule)\')\n    ap.add_argument(\'--limit-pool\', type=int, default=None, help=\'random subset of the pool (smoke tests)\')\n    ap.add_argument(\'--allow-cpu\', action=\'store_true\')\n    a = ap.parse_args()\n    cfgm = dict(MEMBERS[a.member])\n    os.makedirs(a.out, exist_ok=True)\n    summ_path = os.path.join(a.out, f\'summary_{a.member}.json\')\n    if os.path.exists(summ_path):\n        log(f\'{summ_path} exists, nothing to do\')\n        return\n    if not torch.cuda.is_available() and not a.allow_cpu:\n        raise SystemExit(\'no CUDA GPU in this session (pass --allow-cpu only for smoke tests)\')\n    dev = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    T0 = time.time()\n\n    train = pd.read_csv(a.train)\n    train[\'text\'] = train[\'text\'].fillna(\'\').astype(str)\n    pool = pd.read_parquet(a.pool)\n    pool = pool[pool.model.isin(LABELS)].reset_index(drop=True)\n    keep = leak_filter(pool.text.tolist(), train.text.tolist()) & ~pool.text.isin(set(train.text))\n    log(f\'pool {len(pool)}, kept {int(keep.sum())} after the train-similarity filter\')\n    pool = pool[keep].reset_index(drop=True)\n    if a.limit_pool:\n        pool = pool.sample(n=min(a.limit_pool, len(pool)), random_state=0).reset_index(drop=True)\n    lab = {l: i for i, l in enumerate(LABELS)}\n    y_tr, y_pool = train.model.map(lab).values, pool.model.map(lab).values\n\n    model, tok = build_model(a.backbone or cfgm[\'backbone\'], a.member)\n    model.to(dev)\n    ids_tr, ids_pool = encode(tok, train.text), encode(tok, pool.text)\n    log(f\'tokens: pool mean {np.mean([len(x) for x in ids_pool]):.0f}, max {max(len(x) for x in ids_pool)}; \'\n        f\'train mean {np.mean([len(x) for x in ids_tr]):.0f}\')\n\n    # The longest micro-batch first, so an out-of-memory error shows in minute one, not hour three.\n    longest = list(np.argsort([len(x) for x in ids_pool])[-cfgm[\'micro\']:])\n    model.train()\n    enc = tok.pad({\'input_ids\': [ids_pool[i] for i in longest]}, return_tensors=\'pt\').to(dev)\n    with torch.autocast(dev.type, dtype=amp_dtype(dev), enabled=amp_dtype(dev) is not None):\n        model(**enc).logits.float().sum().backward()\n    model.zero_grad(set_to_none=True)\n    if dev.type == \'cuda\':\n        log(f\'memory probe ok: peak {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB\')\n\n    # Stage 1: pool only, class-weighted (inverse frequency, mean 1).\n    cnt = np.bincount(y_pool, minlength=K).astype(float)\n    cw = np.where(cnt > 0, cnt.sum() / np.maximum(cnt, 1) / K, 0.0)\n    info1 = fit(model, tok, ids_pool, y_pool, a.stage1_epochs, cfgm[\'lr1\'], cfgm[\'bs1\'], cfgm[\'micro\'], dev, class_weight=cw,\n                max_secs=a.stage1_hours * 3600 if a.stage1_hours else None, tag=\'stage1\')\n    sd1 = {k: v.detach().to(\'cpu\', copy=True) for k, v in model.state_dict().items()}\n    lg1 = predict(model, tok, ids_tr, dev)\n    ba1 = balanced_accuracy_score(y_tr, lg1.argmax(1))\n    log(f\'stage 1 only, on train: balanced accuracy {ba1:.4f}\')\n\n    # Stage 2 per fold, from the stage-1 weights.\n    folds = np.zeros(len(train), int)\n    for f, (_, va) in enumerate(GroupKFold(n_splits=a.folds).split(train, groups=train.prompt_id)):\n        folds[va] = f\n    oof = np.zeros((len(train), K), np.float32)\n    info2 = []\n    for f in range(a.folds):\n        model.load_state_dict(sd1)\n        tr, va = np.where(folds != f)[0], np.where(folds == f)[0]\n        info2.append(fit(model, tok, [ids_tr[i] for i in tr], y_tr[tr], a.stage2_epochs, a.stage2_lr, a.stage2_bs,\n                         cfgm[\'micro\'], dev, seed=42 + f, tag=f\'fold{f}\'))\n        oof[va] = predict(model, tok, [ids_tr[i] for i in va], dev)\n        log(f\'fold {f}: balanced accuracy {balanced_accuracy_score(y_tr[va], oof[va].argmax(1)):.4f}\')\n    ba = balanced_accuracy_score(y_tr, oof.argmax(1))\n    log(f\'OOF after stage 2: balanced accuracy {ba:.4f}\')\n    np.savez(os.path.join(a.out, f\'oof_{a.member}.npz\'), ids=train.id.astype(str).to_numpy(dtype=str),\n             prompt_id=train.prompt_id.astype(str).to_numpy(dtype=str), fold=folds, y=y_tr,\n             labels=np.array(LABELS, dtype=str), oof=oof, stage1=lg1)\n\n    # Shipped model: stage 2 on the whole train, saved for enc_infer.py, then reloaded the way enc_infer.py loads it.\n    model.load_state_dict(sd1)\n    del sd1\n    info_f = fit(model, tok, ids_tr, y_tr, a.stage2_epochs, a.stage2_lr, a.stage2_bs, cfgm[\'micro\'], dev, seed=42,\n                 tag=\'final\')\n    lg_mem = predict(model, tok, ids_tr[:256], dev)\n    out_dir = os.path.join(a.out, a.member)\n    if MEMBERS[a.member][\'ckpt\']:\n        model.gradient_checkpointing_disable()\n    model.half().save_pretrained(out_dir, safe_serialization=True)\n    tok.save_pretrained(out_dir)\n    del model\n    from transformers import AutoModelForSequenceClassification, AutoTokenizer\n    re_tok = AutoTokenizer.from_pretrained(out_dir)\n    re_model = AutoModelForSequenceClassification.from_pretrained(out_dir, dtype=torch.float32).to(dev).eval()\n    assert [re_model.config.id2label[i] for i in range(re_model.config.num_labels)] == LABELS\n    lg_re = predict(re_model, re_tok, encode(re_tok, train.text[:256]), dev)\n    agree = float((lg_re.argmax(1) == lg_mem.argmax(1)).mean())\n    log(f\'reloaded {out_dir}: top-1 agreement with the in-memory model {agree:.3f} on 256 train rows\')\n    assert agree >= 0.97, agree\n\n    summ = dict(member=a.member, backbone=a.backbone or cfgm[\'backbone\'], recipe=cfgm, pool_used=len(pool),\n                stage1=info1, stage1_only_bacc=ba1, oof_bacc=ba,\n                fold_bacc=[balanced_accuracy_score(y_tr[folds == f], oof[folds == f].argmax(1)) for f in range(a.folds)],\n                stage2=info2, final=info_f, reload_agreement=agree, total_secs=time.time() - T0,\n                device=torch.cuda.get_device_name(dev) if dev.type == \'cuda\' else \'cpu\',\n                torch=torch.__version__, transformers=__import__(\'transformers\').__version__)\n    json.dump(summ, open(summ_path, \'w\'), indent=1, default=float)\n    log(f\'done in {time.time() - T0:.0f}s\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
FILES['08_blend_oof.py'] = '"""Step 8: OOF evidence for the rebuilt encoder members: each member alone, and its gain in the blend (CPU).\n\nWhat: loads the members\' OOF logits from 07 (oof_<member>.npz) and reproduces the 0.620 base\'s n-gram SVC out of fold:\nthe notebook\'s four TF-IDF views and LinearSVC, on the same prompt-grouped folds, fitted on the pool plus the train folds.\nIt then scores, by balanced accuracy and after Sinkhorn as in the notebooks:\n- each member alone;\n- SVC + members at the public notebooks\' OURS weights;\n- a paired bootstrap over prompts, giving P(blend better than SVC alone).\n\nWhy: the base\'s own OOF (LoRAs, DeBERTa) is not published, so the SVC is the only base member we can rebuild out of fold.\nThe gain over SVC alone is an upper bound on the gain over the full 0.620 base, whose stronger members overlap more with\nours.\n\nOutput: <out>/oof_svc.npz (cached; delete it to refit) and <out>/blend_oof.json; the table is printed.\nUsage: python 08_blend_oof.py --train data/train.csv --pool pool_v4.parquet --oof-dir DIR_WITH_oof_npz --out DIR\n"""\nimport argparse\nimport importlib.util\nimport json\nimport os\nimport re\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.sparse import hstack\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics import balanced_accuracy_score\nfrom sklearn.svm import LinearSVC\n\nHERE = os.path.dirname(os.path.abspath(__file__))\n_spec = importlib.util.spec_from_file_location(\'t07\', os.path.join(HERE, \'07_train_encoders.py\'))\nt07 = importlib.util.module_from_spec(_spec)\n_spec.loader.exec_module(t07)\nLABELS, K = t07.LABELS, t07.K\n\n# The base notebook\'s SVC cell, unchanged.\nSMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())\nVIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]\nWEIGHTS = {\'nuridudak 0.624 / mrpicklerick 0.624\': dict(modernbert=0.25, desklib=0.10),\n           \'nuridudak 0.626\': dict(modernbert=0.40, desklib=0.10),\n           \'mrpicklerick 0.627\': dict(modernbert=0.30, desklib=0.05)}\n\n\ndef shape_text(t):\n    return re.sub(r"[^\\W\\d_]+", lambda m: "X" if m[0].isupper() else "x", t)\n\n\ndef small_words_text(t):\n    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"\n                    for w in re.findall(r"\\w+|[^\\w\\s]", t))\n\n\ndef svc_scores(fit_texts, fit_y, pred_texts):\n    A, B = [], []\n    for view, lo, hi, max_features, weight in VIEWS:\n        prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)\n        analyzer = "char" if view == "shape" else "word" if view == "small" else view\n        opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,\n                    max_features=max_features, dtype=np.float32)\n        if analyzer == "word":\n            opts["token_pattern"] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n        vec = TfidfVectorizer(**opts)\n        A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)\n        B.append(vec.transform([prep(t) for t in pred_texts]) * weight)\n    svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)\n    return svc.decision_function(hstack(B, format="csr"))\n\n\ndef softmax(x):\n    e = np.exp(x - x.max(1, keepdims=True))\n    return e / e.sum(1, keepdims=True)\n\n\ndef sinkhorn(P, iters=300):\n    c = np.ones(K)\n    target = np.full(K, len(P) / K)\n    for _ in range(iters):\n        R = P * c\n        R /= R.sum(1, keepdims=True)\n        c *= target / R.sum(0)\n    R = P * c\n    return R / R.sum(1, keepdims=True)\n\n\ndef blend(parts, rows=slice(None)):\n    """parts: list of (weight, probabilities). Sinkhorn over the rows scored together, as on the test set."""\n    L = sum(w * np.log(np.clip(P[rows], 1e-9, 1)) for w, P in parts)\n    return sinkhorn(softmax(L)).argmax(1)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\'--train\', required=True)\n    ap.add_argument(\'--pool\', required=True)\n    ap.add_argument(\'--oof-dir\', required=True)\n    ap.add_argument(\'--out\', required=True)\n    ap.add_argument(\'--boot\', type=int, default=2000)\n    a = ap.parse_args()\n    os.makedirs(a.out, exist_ok=True)\n    train = pd.read_csv(a.train)\n    train[\'text\'] = train[\'text\'].fillna(\'\').astype(str)\n    lab = {l: i for i, l in enumerate(LABELS)}\n    y = train.model.map(lab).values\n    members = {}\n    for n in (\'modernbert\', \'desklib\'):\n        p = os.path.join(a.oof_dir, f\'oof_{n}.npz\')\n        if os.path.exists(p):\n            z = np.load(p)\n            assert list(z[\'ids\']) == train.id.astype(str).tolist() and list(z[\'labels\']) == LABELS\n            members[n] = dict(P=softmax(z[\'oof\'].astype(np.float64)), P1=softmax(z[\'stage1\'].astype(np.float64)))\n            folds = z[\'fold\']\n    if not members:   # same folds as 07\n        from sklearn.model_selection import GroupKFold\n        folds = np.zeros(len(train), int)\n        for f, (_, va) in enumerate(GroupKFold(n_splits=5).split(train, groups=train.prompt_id)):\n            folds[va] = f\n\n    svc_path = os.path.join(a.out, \'oof_svc.npz\')\n    if os.path.exists(svc_path):\n        S = np.load(svc_path)[\'scores\']\n    else:\n        pool = pd.read_parquet(a.pool)\n        pool = pool[pool.model.isin(LABELS)].reset_index(drop=True)\n        keep = t07.leak_filter(pool.text.tolist(), train.text.tolist()) & ~pool.text.isin(set(train.text))\n        pool = pool[keep].reset_index(drop=True)   # the same honest pool as 07\'s stage 1\n        S = np.zeros((len(train), K))\n        for f in range(folds.max() + 1):\n            tr, va = folds != f, folds == f\n            S[va] = svc_scores(train.text[tr].tolist() + pool.text.tolist(),\n                               np.r_[y[tr], pool.model.map(lab).values], train.text[va].tolist())\n            t07.log(f\'svc fold {f}: {balanced_accuracy_score(y[va], S[va].argmax(1)):.4f}\')\n        np.savez(svc_path, scores=S, fold=folds, ids=train.id.astype(str).to_numpy(dtype=str))\n    Psvc = softmax(S)\n\n    res = {\'svc alone (argmax)\': balanced_accuracy_score(y, Psvc.argmax(1)),\n           \'svc alone + sinkhorn\': balanced_accuracy_score(y, blend([(1.0, Psvc)]))}\n    for n, m in members.items():\n        res[f\'{n} alone (argmax)\'] = balanced_accuracy_score(y, m[\'P\'].argmax(1))\n        res[f\'{n} stage-1 only (argmax)\'] = balanced_accuracy_score(y, m[\'P1\'].argmax(1))\n    if len(members) == 2:\n        res[\'modernbert+desklib 1:1 + sinkhorn\'] = balanced_accuracy_score(\n            y, blend([(1.0, members[\'modernbert\'][\'P\']), (1.0, members[\'desklib\'][\'P\'])]))\n    configs = {name: [(1.0, Psvc)] + [(w[n], members[n][\'P\']) for n in members] for name, w in WEIGHTS.items()}\n    configs[\'svc + modernbert 1.0 + desklib 0.5\'] = [(1.0, Psvc)] + [(dict(modernbert=1.0, desklib=0.5)[n], members[n][\'P\'])\n                                                                    for n in members]\n    base_pred = blend([(1.0, Psvc)])\n    preds = {name: blend(parts) for name, parts in configs.items()}\n    for name, pr in preds.items():\n        res[f\'svc + OURS [{name}] + sinkhorn\'] = balanced_accuracy_score(y, pr)\n\n    # Paired bootstrap over prompts: resample the 150 prompts, rescore both (predictions fixed, Sinkhorn already done).\n    rng = np.random.default_rng(0)\n    groups = train.prompt_id.values\n    ug = np.unique(groups)\n    gidx = {g: np.where(groups == g)[0] for g in ug}\n    boot = {}\n    for name, pr in preds.items():\n        d = []\n        for _ in range(a.boot):\n            idx = np.concatenate([gidx[g] for g in rng.choice(ug, len(ug))])\n            d.append(balanced_accuracy_score(y[idx], pr[idx]) - balanced_accuracy_score(y[idx], base_pred[idx]))\n        d = np.array(d)\n        boot[name] = dict(mean_gain=float(d.mean()), p_better=float((d > 0).mean()),\n                          ci90=[float(np.quantile(d, 0.05)), float(np.quantile(d, 0.95))])\n    for k, v in res.items():\n        print(f\'{k:60s} {v:.4f}\')\n    for k, v in boot.items():\n        print(f\'bootstrap [{k}] gain {v["mean_gain"]:+.4f}, 90% CI {v["ci90"][0]:+.4f}..{v["ci90"][1]:+.4f}, \'\n              f\'P(better) {v["p_better"]:.3f}\')\n    json.dump(dict(scores=res, bootstrap=boot), open(os.path.join(a.out, \'blend_oof.json\'), \'w\'), indent=1)\n\n\nif __name__ == \'__main__\':\n    main()\n'
for name, text in FILES.items():
    path = os.path.join(CODE, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w").write(text)
CODE_SHA = 'ea4a83c263688cda'
print("wrote", len(FILES), "files to", CODE, "| code", CODE_SHA)


In [ ]:
UP = "/content/upload"


def upload(msg, **progress):
    """Upload every finished output (no resume points) and progress.json as a new version of OUTPUTS; 3 tries."""
    if os.path.exists(UP):
        shutil.rmtree(UP)
    os.makedirs(UP)
    for f in sorted(os.listdir(WORK)):
        src = f"{WORK}/{f}"
        if os.path.isdir(src) and f != "state":
            shutil.copytree(src, f"{UP}/{f}")
        elif f.endswith((".npz", ".json")) and f != "progress.json":
            shutil.copy(src, UP)
    progress.update(event=msg, gpu=GPU, code_sha256=CODE_SHA, time_utc=time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()))
    json.dump(progress, open(f"{UP}/progress.json", "w"), indent=1)
    json.dump({"title": OUTPUTS.split("/")[1], "id": OUTPUTS, "licenses": [{"name": "CC-BY-SA-4.0"}]},
              open(f"{UP}/dataset-metadata.json", "w"))
    for attempt in range(3):
        exists = subprocess.run(["kaggle", "datasets", "status", OUTPUTS], capture_output=True).returncode == 0
        cmd = (["kaggle", "datasets", "version", "-p", UP, "-m", msg, "-r", "zip"] if exists
               else ["kaggle", "datasets", "create", "-p", UP, "-r", "zip"])
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            print(f"uploaded ({msg}) to https://www.kaggle.com/datasets/{OUTPUTS}", flush=True)
            return
        print(f"upload attempt {attempt + 1} failed: {(r.stdout + r.stderr).strip()[-300:]}", flush=True)
        time.sleep(60)   # a just-created dataset takes a moment before it accepts versions
    print("WARNING: upload failed; the files are safe on Drive in", WORK, flush=True)


def done(unit):
    f = dict(stage1="summary_stage1.json", full="summary_full.json", merge="summary_olmo13b.json").get(unit, f"oof_{unit}.npz")
    return os.path.exists(f"{WORK}/{f}")


In [ ]:
hours = {}
for unit in UNITS:
    if done(unit):
        print(unit, "already done")
        continue
    left = sum(EST_H[u] for u in UNITS if not done(u))
    print(f"\n=== {unit} (about {EST_H[unit]} h on a T4; {left:.1f} h of units left)", flush=True)
    t = time.time()
    cmd = [sys.executable, "-u", f"{CODE}/09_train_olmo13b.py", "--unit", unit, "--train", f"{DATA}/train.csv",
           "--pool", f"{DATA}/pool_v4.parquet", "--base", BASE, "--out", WORK]
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "TOKENIZERS_PARALLELISM": "false",
                              "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
    for line in p.stdout:
        if "warn" not in line.lower() and "UNEXPECTED" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{unit} failed (exit {p.returncode}); Run all again to resume from the last save")
    hours[unit] = round((time.time() - t) / 3600, 2)
    summary = json.load(open(f"{WORK}/summary_olmo13b.json")) if os.path.exists(f"{WORK}/summary_olmo13b.json") else None
    upload(f"{unit} done", units_done=[u for u in UNITS if done(u)], hours_this_session=hours, summary=summary)
print("all units done:", [u for u in UNITS if done(u)])
if os.path.exists(f"{WORK}/summary_olmo13b.json"):
    print(open(f"{WORK}/summary_olmo13b.json").read()[:3000])
